# 08 — Safety stock, ROP, EOQ, ABC

Lead time and costs are scenario parameters. Change `service_level` and recompute.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.utils.logging_config import setup_logging
from src.utils.helpers import load_config, load_model_config, resolve_path
setup_logging("INFO")
CONFIG = load_config()
print("Independent M5-schema project. DATA_DIR =", resolve_path(CONFIG["paths"]["data_dir"]))


In [ ]:
import pandas as pd
from src.inventory.engine import run_inventory_engine, run_scenario
from src.features.inventory_features import attach_cost_assumptions, simulate_on_hand_inventory

profile = pd.read_parquet(resolve_path(CONFIG["paths"]["processed_dir"]) / "demand_profile.parquet")
if "lead_time_days" not in profile.columns:
    profile = attach_cost_assumptions(profile, CONFIG)
    profile = simulate_on_hand_inventory(profile)
base = run_inventory_engine(profile, CONFIG)
print(base["stockout_risk"].value_counts())
print(base["abc_class"].value_counts())
base[["item_id", "store_id", "safety_stock", "reorder_point", "eoq", "abc_class", "recommended_action"]].head()


In [ ]:
tight = run_scenario(profile, CONFIG, service_level=0.99, lead_time_days=10, ordering_cost=75, holding_cost_rate=0.25)
compare = base.merge(tight, on=["item_id", "store_id"], suffixes=("_95", "_99"))
print((compare["safety_stock_99"] - compare["safety_stock_95"]).mean(), "mean SS increase at 99% / LT=10")
